# Tutorial: Affiliation Pipeline

This notebook serves as a **guide, example, and reference** for the affiliation pipeline.

It shows the standard workflow from a raw CSV file to the final ROR result:

`CSV → prepare columns → split affiliations → remove locations → query ROR → review results → save CSV`

Important:

- Original raw columns are never overwritten.
- The ROR API receives `affiliation_part_raw`, meaning the affiliation text in its most original form possible.
- `affiliation_clean` is used for the cleaned organizational structure.
- `ror_original` is optional and is not treated as error-free ground truth.
- `review` only means that a case should deliberately be checked more closely.
- `unmatched` is not automatically an error. Many companies, GmbHs, or small institutions simply do not have a ROR record.
- Fully automatic assignment for every possible affiliation is not the goal. The pipeline should substantially reduce manual work and make problematic cases visible.

## 0. Expected project structure

By default, the following structure is expected:

```text
affiliation-pipeline/
├── affiliation_pipeline/
│   ├── __init__.py
│   ├── preprocessing.py
│   ├── locations.py
│   └── ror_api.py
├── data/
│   ├── affilation_test.csv
│   ├── cities500/
│   └── alternateNamesV2/
├── jupyter/
│   └── tutorial_affiliation_pipeline_en_v1.ipynb
├── .env
└── requirements.txt
```

The file `affilation_test.csv` is only the **default dataset for this tutorial**.

To use your own data:

1. Copy the CSV file into the `data/` folder.
2. In the next configuration cell, change only `DATA_FILENAME`.
3. If the affiliation column has a different name, also change `SOURCE_AFFILIATION_COLUMN`.

The GeoNames data is required locally and normally should **not** be committed to the Git repository.

## 1. Central configuration

This cell is normally the only place that needs to be adapted for a new dataset.

### Required

- `DATA_FILENAME`: name of the input CSV file inside the `data/` folder.
- `SOURCE_AFFILIATION_COLUMN`: column containing the original affiliation text.

### Optional

- `ORIGINAL_ROR_COLUMN`: existing ROR column. Set to `None` if there is no such column.
- `SPLIT_AFFILIATIONS`: use `True` only if multiple affiliations inside one cell are separated by a fixed delimiter.
- `AFFILIATION_DELIMITER`: `;` in our test data.
- `INCLUDE_LOCATION_DIAGNOSTICS`: additional diagnostic fields for city/country recognition.
- `RUN_ROR_API`: can be set to `False` for preprocessing-only tests.

In [ ]:
DATA_FILENAME = "affilation_test.csv"

SOURCE_AFFILIATION_COLUMN = "affilation"
STANDARD_AFFILIATION_COLUMN = "affiliation"

ORIGINAL_ROR_COLUMN = "ror_original"

SPLIT_AFFILIATIONS = True
AFFILIATION_DELIMITER = ";"

INCLUDE_LOCATION_DIAGNOSTICS = True
REMOVE_ADDRESSES = True

RUN_ROR_API = True

OUTPUT_FILENAME = "affiliation_result.csv"

RANDOM_STATE = 42

## 2. Imports and project path

The notebook can either be located directly in the project folder or inside the `jupyter/` subfolder.

`PROJECT_ROOT` is therefore determined automatically.

The most important pipeline functions are:

- `load_csv()` – load a CSV file.
- `preview_dataframe()` – inspect column positions/names and a random sample.
- `rename_columns()` – rename columns without tying the code to a specific source file.
- `split_column()` – split one column either into new rows or new columns.
- `load_location_resources()` – load country and GeoNames resources.
- `process_locations()` – extract city, country, postcode, and address information and create `affiliation_clean`.
- `norm_country()` / `norm_city()` – normalize individual country or city names.
- `RORAffiliationProcessor` – ROR requests, cache, fallback logic, statuses, and structured clean columns.
- `save_csv()` – save the result.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "jupyter":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))

from affiliation_pipeline.preprocessing import (
    load_csv,
    preview_dataframe,
    rename_columns,
    save_csv,
    split_column,
)

from affiliation_pipeline.locations import (
    load_location_resources,
    norm_city,
    norm_country,
    process_locations,
)

from affiliation_pipeline.ror_api import RORAffiliationProcessor

DATA_FILE = PROJECT_ROOT / "data" / DATA_FILENAME
CACHE_FILE = PROJECT_ROOT / "data" / "ror_api_cache.json"
CHECKPOINT_FILE = PROJECT_ROOT / "data" / "ror_api_checkpoint.csv"

print("Project:", PROJECT_ROOT)
print("Input file:", DATA_FILE)

## 3. Load the CSV – `load_csv()`

`load_csv()` reads the file into a DataFrame.

For this project, `;` is the usual **CSV field separator**.

This is different from a semicolon used inside an affiliation field to separate multiple affiliations. If a field itself contains semicolons, the CSV must be quoted correctly.

Standard usage:

```python
df = load_csv(DATA_FILE)
```

If an external CSV uses `,` as the field separator, for example, the corresponding pandas argument can be passed through:

```python
df = load_csv(DATA_FILE, sep=",")
```

In [ ]:
df = load_csv(DATA_FILE)

print(f"Loaded rows: {len(df):,}")
print(f"Columns: {len(df.columns)}")

preview_dataframe(
    df,
    n=5,
    random_state=RANDOM_STATE,
)

## 4. Standardize column names – `rename_columns()`

The pipeline should internally work with a consistent column called `affiliation`.

Our test file intentionally uses `affilation`, so it is renamed here.

`rename_columns()` receives a mapping:

```python
{
    "old_name": "new_name"
}
```

Multiple columns can be renamed at the same time.

This step is **optional** if the source file already uses the desired column names.

In [ ]:
rename_map = {}

if (
    SOURCE_AFFILIATION_COLUMN != STANDARD_AFFILIATION_COLUMN
    and SOURCE_AFFILIATION_COLUMN in df.columns
):
    rename_map[SOURCE_AFFILIATION_COLUMN] = STANDARD_AFFILIATION_COLUMN

if rename_map:
    df = rename_columns(df, rename_map)

if STANDARD_AFFILIATION_COLUMN not in df.columns:
    raise KeyError(
        f"The affiliation column {STANDARD_AFFILIATION_COLUMN!r} was not found."
    )

print("Affiliation column used:", STANDARD_AFFILIATION_COLUMN)

## 5. Multiple affiliations in one cell – `split_column()`

Some source data contains several affiliations in one row, for example:

```text
University A;Company B;Institute C
```

In that case, each affiliation should get its own row.

### Option A: split into rows

For the ROR workflow, this is normally the correct option:

```python
split_column(
    df,
    column="affiliation",
    delimiter=";",
    into="rows",
    output_column="affiliation_part_raw",
)
```

This means:

- `affiliation` remains unchanged,
- `affiliation_part_raw` is created as a new column,
- all other columns are copied to each resulting row,
- `original_row_id` can be stored so the source row remains traceable.

### Option B: split into columns

For other data-cleaning tasks, the same helper can split horizontally:

```python
split_column(
    df,
    column="example",
    delimiter="|||",
    into="columns",
    output_prefix="example_part",
)
```

This automatically creates `example_part_1`, `example_part_2`, ... up to the maximum number of parts actually needed in the dataset.

For the ROR pipeline, **Option A** is used below.

In [ ]:
if SPLIT_AFFILIATIONS:
    df_parts = split_column(
        df,
        column=STANDARD_AFFILIATION_COLUMN,
        delimiter=AFFILIATION_DELIMITER,
        into="rows",
        output_column="affiliation_part_raw",
        row_id_column="original_row_id",
        drop_empty=True,
    )
else:
    df_parts = df.copy()
    df_parts["affiliation_part_raw"] = (
        df_parts[STANDARD_AFFILIATION_COLUMN]
        .astype("string")
        .str.strip()
    )

print(f"Rows before: {len(df):,}")
print(f"Rows after preparation: {len(df_parts):,}")

display(
    df_parts[
        [
            column
            for column in [
                "original_row_id",
                STANDARD_AFFILIATION_COLUMN,
                "affiliation_part_raw",
            ]
            if column in df_parts.columns
        ]
    ].head(10)
)

## 6. Load GeoNames and country resources – `load_location_resources()`

This function builds the reference data for countries and cities.

For city recognition, two GeoNames files are required:

- [`cities500.zip`](https://download.geonames.org/export/dump/cities500.zip)  
  Contains places with at least 500 inhabitants as well as other relevant geographic entries.

- [`alternateNamesV2.zip`](https://download.geonames.org/export/dump/alternateNamesV2.zip)  
  Contains alternative names, translations, and spelling variants for GeoNames entries.

All GeoNames download files are also available in the official download directory:

<https://download.geonames.org/export/dump/>

After downloading, extract the ZIP files and place them in the following folders:

```text
data/
├── cities500/
│   └── cities500.txt
└── alternateNamesV2/
    └── alternateNamesV2.txt

The GeoNames files are not stored in the Git repository and therefore need to be downloaded locally.

By default, the data is loaded from the project's `data/` folder.

Important optional parameters:

- `data_dir=` – a different base folder for GeoNames files.
- `cities_path=` – explicit path to the `cities500` file.
- `alternate_names_path=` – explicit path to `alternateNamesV2`.
- `language_codes_path=` – optional path to GeoNames language codes.
- `locales=("en", "de")` – language variants for country names.
- `additional_country_aliases={...}` – custom additional country aliases.
- `show_progress=True` – show progress while loading.
- `alternate_names_chunksize=` – chunk size when reading large alternate-name files.

For the normal project structure, this is sufficient:

```python
resources = load_location_resources()
```

Loading can take some time because `alternateNamesV2` is large.

In [ ]:
resources = load_location_resources(
    show_progress=True,
)

## 7. Test individual countries or cities – `norm_country()` and `norm_city()`

These two functions are **not required for the full pipeline**. They are small helper functions for tests or manual checks.

Examples:

```python
norm_country("Austria", resources)
norm_country("AT", resources)

norm_city("Vienna", resources)
norm_city("Wien", resources)
```

Recognition is intentionally conservative. A name should not be removed from an organization merely because it also happens to be a place name.

In [ ]:
print("Country:", norm_country("Austria", resources))
print("Country code:", norm_country("AT", resources))
print("City Vienna:", norm_city("Vienna", resources))
print("City Wien:", norm_city("Wien", resources))

## 8. Location cleaning – `process_locations()`

This is the central location-processing function.

It receives:

### Required

- the DataFrame,
- `column=` – the column to clean,
- `resources=` – the previously loaded country/GeoNames resources.

### Optional

- `output_column="affiliation_clean"` – name of the cleaned affiliation column.
- `output_columns=[...]` – which extracted location columns should actually be created.
- `remove_addresses=True` – remove recognized address segments from `affiliation_clean`.
- `include_diagnostics=True` – add extra status/diagnostic columns.
- `show_progress=True` – show a `tqdm` progress bar.

Possible regular location outputs are:

- `address_raw`
- `postcode_raw`
- `city_raw`
- `city_norm`
- `country_raw`
- `country_norm`

With `include_diagnostics=True`, additional diagnostic fields are added, such as `country_iso2`, `country_status`, `city_status`, `city_entity_status`, `city_match_type`, and `city_source_segment`.

Important: `affiliation_part_raw` remains unchanged. `affiliation_clean` is a **new** column.

In [ ]:
location_output_columns = [
    "address_raw",
    "postcode_raw",
    "city_raw",
    "city_norm",
    "country_raw",
    "country_norm",
]

df_clean = process_locations(
    df_parts,
    column="affiliation_part_raw",
    resources=resources,
    output_column="affiliation_clean",
    output_columns=location_output_columns,
    remove_addresses=REMOVE_ADDRESSES,
    include_diagnostics=INCLUDE_LOCATION_DIAGNOSTICS,
    show_progress=True,
)

preview_columns = [
    column
    for column in [
        "id",
        "affiliation_part_raw",
        "affiliation_clean",
        "address_raw",
        "postcode_raw",
        "city_raw",
        "city_norm",
        "country_raw",
        "country_norm",
        "country_iso2",
        "country_status",
        "city_status",
    ]
    if column in df_clean.columns
]

display(
    df_clean[preview_columns].sample(
        n=min(15, len(df_clean)),
        random_state=RANDOM_STATE,
    )
)

## 9. Optional: use only preprocessing, without the ROR API

The pipeline can also be used without ROR.

If only location cleaning is needed:

```python
RUN_ROR_API = False
```

Then `df_clean` can be saved directly or processed further.

The following API cells in this tutorial are only executed when `RUN_ROR_API = True`.

## 10. ROR configuration and `.env`

By default, the ROR processor uses:

- `affiliation_part_raw` as API input,
- `affiliation_clean` for later structuring,
- `country_iso2` / `country_norm` as additional information for exact fallbacks.

### `.env`

If a ROR Client ID is used:

```text
ROR_CLIENT_ID=...
```

The `.env` file remains local and should not be committed to GitHub.

The pipeline also works without a Client ID, but it uses a much larger request interval.

### Cache

`ror_api_cache.json` stores requests that have already been made. Repeated identical requests therefore do not have to be sent to the API again.

### Checkpoint

`ror_api_checkpoint.csv` is saved regularly during longer runs. It is only a safety copy of the intermediate state.

The cache and checkpoint can be deleted when a completely fresh test is intentionally required.

## 11. Create the `RORAffiliationProcessor`

### Required or central parameters

- `api_input_col` – text sent to ROR. In our workflow: `affiliation_part_raw`.
- `clean_col` – cleaned text used for later structuring. In our workflow: `affiliation_clean`.

### Frequently used optional parameters

- `original_ror_col` – existing ROR column or `None`.
- `country_code_col` – ISO-2 country column, normally `country_iso2`.
- `country_name_col` – normalized country, normally `country_norm`.
- `clean_output_prefix` – prefix for `affiliation_clean_1`, `_2`, ...
- `output_ror_col` – `ror_after_api` by default.
- `cache_file` – custom cache path.
- `max_fallback_queries` – maximum number of additional exact-fallback requests per problematic row.
- `timeout` – request timeout in seconds.
- `max_retries` – number of retry attempts.
- `min_request_interval` – manually override the delay between requests.
- `checkpoint_every` – number of processed rows between cache/checkpoint saves.
- `chosen_locator_threshold` – internal plausibility threshold for `chosen=true` results.

The defaults should normally not be changed without a concrete reason.

- `min_request_interval` – manually override the delay between API requests.  
  ROR currently allows up to 2,000 requests per 5 minutes. A value of about `0.25` seconds is a conservative practical setting for this limit.  
  Once the announced Client ID limits are activated, the standard limit will remain up to 2,000 requests per 5 minutes with a valid Client ID, while requests without a Client ID will be limited to 50 requests per 5 minutes; in that case, an interval of about `6.5` seconds is appropriate.

In [ ]:
original_ror_col = (
    ORIGINAL_ROR_COLUMN
    if ORIGINAL_ROR_COLUMN
    and ORIGINAL_ROR_COLUMN in df_clean.columns
    else None
)

processor = RORAffiliationProcessor(
    api_input_col="affiliation_part_raw",
    clean_col="affiliation_clean",
    original_ror_col=original_ror_col,
    country_code_col=(
        "country_iso2"
        if "country_iso2" in df_clean.columns
        else None
    ),
    country_name_col=(
        "country_norm"
        if "country_norm" in df_clean.columns
        else None
    ),
    clean_output_prefix="affiliation_clean",
    output_ror_col="ror_after_api",
    cache_file=CACHE_FILE,
)

print(
    "ROR Client ID:",
    "available" if processor.client_id else "not available",
)
print(
    "Request interval:",
    f"{processor.min_request_interval:.2f} seconds",
)
print(
    "Rows for the potential API run:",
    f"{len(df_clean):,}",
)

## 12. Optional: inspect exact-fallback candidates

`generate_exact_candidates()` makes **no API request**.

The function only shows which organization strings the processor would try as exact-fallback candidates when there is no useful `chosen=true` result.

This is mainly useful for debugging.

Here, `limit=3` only means that at most three candidates are displayed.

In [ ]:
candidate_preview = df_clean[
    ["affiliation_clean"]
].sample(
    n=min(15, len(df_clean)),
    random_state=RANDOM_STATE,
).copy()

candidate_preview["exact_candidates"] = (
    candidate_preview["affiliation_clean"]
    .map(
        lambda value: processor.generate_exact_candidates(
            value,
            limit=3,
        )
    )
)

display(candidate_preview)

## 13. Run the ROR API – `processor.process()`

The normal full run is:

```python
df_ror = processor.process(
    df_clean,
    checkpoint_file=CHECKPOINT_FILE,
    show_progress=True,
)
```

If `max_rows` is not specified, **all open rows** are processed.

### Important optional parameters

- `max_rows=10` – only for small tests; omit it for a production/full run.
- `statuses=[...]` – process only selected statuses again.
- `force=True` – ignore the status filter and process selected rows again.
- `bypass_cache=True` – deliberately bypass the request cache.
- `checkpoint_file=...` – periodically save intermediate state.
- `show_progress=True` – show a progress bar.

Normal continuation after an interrupted run:

```python
processor.process(df, checkpoint_file=..., show_progress=True)
```

The cache prevents unnecessary repeated identical requests.

`bypass_cache=True` should only be used when genuinely fresh API responses are required.

In [ ]:
if RUN_ROR_API:
    df_ror = processor.process(
        df_clean,
        checkpoint_file=CHECKPOINT_FILE,
        show_progress=True,
    )
else:
    df_ror = df_clean.copy()
    print("ROR API was skipped.")

## 14. What does the ROR processor create?

Typical new fields:

- `affiliation_clean_1`, `affiliation_clean_2`, ... – structured organization parts.
- `ror_after_api` – newly determined ROR.
- `ror_status` – for example `active` or `inactive`.
- `status` – broad pipeline decision.
- `status_detail` – more specific reason.
- `ror_successor_id`, `ror_successor_name` – for historical/inactive records when a successor is known.

For a safe active ROR match, the current ROR primary name is written to `affiliation_clean_1`.

Example:

```text
Land and Water, CSIRO
```

can become:

```text
affiliation_clean_1 = Commonwealth Scientific and Industrial Research Organisation
affiliation_clean_2 = Land and Water
```

`affiliation_clean` itself remains unchanged as the output of the previous cleaning step.

In [ ]:
if RUN_ROR_API:
    clean_split_columns = sorted(
        [
            column
            for column in df_ror.columns
            if column.startswith("affiliation_clean_")
            and column[len("affiliation_clean_"):].isdigit()
        ],
        key=lambda column: int(
            column[len("affiliation_clean_"):]
        ),
    )

    result_columns = [
        column
        for column in [
            "id",
            "affiliation_part_raw",
            "affiliation_clean",
            *clean_split_columns,
            ORIGINAL_ROR_COLUMN,
            "ror_after_api",
            "ror_status",
            "status",
            "status_detail",
            "ror_successor_id",
            "ror_successor_name",
        ]
        if column in df_ror.columns
    ]

    display(
        df_ror[result_columns].sample(
            n=min(25, len(df_ror)),
            random_state=RANDOM_STATE,
        )
    )

## 15. Understanding the status fields

The main statuses are intentionally kept small.

### `matched`

The pipeline has enough evidence for an automatic ROR assignment.

Common details:

- `affiliation_chosen` – ROR Affiliation Search returned `chosen=true` and the result was plausible.
- `exact_name_fallback` – a unique exact name was found through fallback.
- `exact_name_country_fallback` – an exact name was additionally disambiguated using the country.
- `alternative_name_match` – an exact ROR name variant/alias was recognized.
- `original_ror_validated` – an existing ROR could be confirmed by the current evidence.

### `unmatched`

No sufficiently good ROR was found.

Typical detail:

- `no_exact_name_match`

This is not automatically a problem. Many companies or small organizations simply do not exist in ROR.

### `review`

There is a reason to inspect the case manually.

Typical details:

- `chosen_name_mismatch` – ROR returned `chosen=true`, but the organization name was not plausibly present in the original text.
- `multiple_organizations` – several independent organizations may be present in the entry.
- `ambiguous_exact_name` – multiple exact ROR candidates are possible.
- `ror_not_unique` – API ROR and existing `ror_original` differ.
- `exact_inactive_match` – exact match to an inactive historical ROR.

### `api_error` / `rate_limited`

Technical error or rate limit. These cases can be processed again later.

In [ ]:
if RUN_ROR_API:
    status_summary = (
        df_ror[
            ["status", "status_detail"]
        ]
        .value_counts(dropna=False)
        .rename("count")
        .reset_index()
    )

    display(status_summary)

## 16. Inspect review cases

These are the most important candidates for manual inspection.

The goal is **not** to eliminate `review` completely. A small review share is preferable to aggressive false positives.

In [ ]:
if RUN_ROR_API:
    review_cases = df_ror.loc[
        df_ror["status"].eq("review"),
        result_columns,
    ]

    print(f"Review cases: {len(review_cases):,}")
    display(review_cases)

## 17. Inspect unmatched cases

For `unmatched`, first check whether the organization actually exists in ROR at all.

Especially for:

- GmbHs,
- small companies,
- local institutions,
- very new organizations,

a missing ROR can be completely correct.

In [ ]:
if RUN_ROR_API:
    unmatched_cases = df_ror.loc[
        df_ror["status"].eq("unmatched"),
        result_columns,
    ].copy()

    print(f"Unmatched cases: {len(unmatched_cases):,}")
    display(unmatched_cases)

## 18. Filter technical failures

If a long API run was interrupted, these statuses are particularly relevant:

- `api_error`
- `rate_limited`
- `pending`

They can be processed again later without touching all successful rows.

In [ ]:
if RUN_ROR_API:
    retry_cases = df_ror.loc[
        df_ror["status"].isin(
            ["pending", "api_error", "rate_limited"]
        ),
        result_columns,
    ]

    print(f"Technical cases to retry: {len(retry_cases):,}")
    display(retry_cases)

## 19. Compare `ror_original` – but do not treat it as ground truth

An existing `ror_original` is very useful for quality control, but it can itself be wrong.

Therefore:

- identical IDs → strong consistency signal,
- different IDs → inspect more closely,
- existing original ROR → do not copy blindly.

ROR URLs also need to be normalized before comparison, because for example:

```text
ror.org/04mte1k06
```

and

```text
https://ror.org/04mte1k06
```

refer to the same ROR ID.

In [ ]:
def normalize_ror_for_comparison(value):
    if pd.isna(value):
        return pd.NA

    text = str(value).strip()

    if not text:
        return pd.NA

    text = text.removeprefix("https://")
    text = text.removeprefix("http://")
    text = text.removeprefix("www.")

    return text.casefold()


if (
    RUN_ROR_API
    and ORIGINAL_ROR_COLUMN
    and ORIGINAL_ROR_COLUMN in df_ror.columns
):
    comparison = df_ror[
        [
            column
            for column in [
                "id",
                "affiliation_part_raw",
                ORIGINAL_ROR_COLUMN,
                "ror_after_api",
                "status",
                "status_detail",
            ]
            if column in df_ror.columns
        ]
    ].copy()

    comparison["ror_original_norm"] = (
        comparison[ORIGINAL_ROR_COLUMN]
        .map(normalize_ror_for_comparison)
    )
    comparison["ror_after_api_norm"] = (
        comparison["ror_after_api"]
        .map(normalize_ror_for_comparison)
    )

    both_present = comparison.dropna(
        subset=[
            "ror_original_norm",
            "ror_after_api_norm",
        ]
    )

    same = (
        both_present["ror_original_norm"]
        == both_present["ror_after_api_norm"]
    )

    print("Rows with both RORs:", len(both_present))
    print("Same ROR:", int(same.sum()))
    print("Different ROR:", int((~same).sum()))

    display(
        both_present.loc[
            ~same
        ]
    )
else:
    print("No original ROR column is available for comparison.")

## 20. Save the result – `save_csv()`

`save_csv()` saves the DataFrame as a CSV file.

With the normal project structure, it writes into the `data/` folder.

Example:

```python
save_csv(
    df_ror,
    "affiliation_result.csv",
    overwrite=True,
)
```

`overwrite=True` is intentionally explicit. Without this permission, an existing result file should not be overwritten accidentally.

In [ ]:
save_csv(
    df_ror,
    OUTPUT_FILENAME,
    overwrite=True,
)

# Common variants

## A. Use your own CSV file

Only change the configuration above:

```python
DATA_FILENAME = "my_file.csv"
SOURCE_AFFILIATION_COLUMN = "organization"
```

Copy the file into `data/` first.

---

## B. No existing ROR column

```python
ORIGINAL_ROR_COLUMN = None
```

The ROR processor still works normally.

---

## C. Already one affiliation per row

```python
SPLIT_AFFILIATIONS = False
```

No semicolon split is performed. The existing affiliation text is copied directly to `affiliation_part_raw`.

---

## D. Different separator inside a cell

Example:

```python
AFFILIATION_DELIMITER = "|||"
```

`split_column()` treats the delimiter literally, not as a regular expression.

---

## E. Test only location cleaning

```python
RUN_ROR_API = False
```

This avoids API requests and waiting time.

---

## F. Test only a few rows with ROR

Temporarily add to the `process()` call:

```python
max_rows=10
```

Remove it again, or use `None`, for the full run.

---

## G. Deliberately start a completely fresh API run

1. Delete the cache file or choose a new cache filename.
2. Delete the checkpoint if necessary.
3. Run the pipeline again.

`bypass_cache=True` is also possible, but should not be the default.

---

## H. Continue after a technical error

Normally, calling `process()` again is sufficient.

By default, open or technical statuses such as `pending`, `api_error`, and `rate_limited` are processed again.

---

## I. Add custom country aliases

When loading location resources:

```python
resources = load_location_resources(
    additional_country_aliases={
        "my alias": "AT",
    }
)
```

The value on the right is an ISO-2 country code.

---

## J. Create fewer location output columns

Not all six location columns need to be created:

```python
df_clean = process_locations(
    df_parts,
    column="affiliation_part_raw",
    resources=resources,
    output_columns=[
        "city_norm",
        "country_norm",
    ],
)
```

`affiliation_clean` is still created.

# Quick reference for the main functions

| Function | Purpose | Required | Frequently optional |
|---|---|---|---|
| `load_csv()` | Load CSV | file path | `sep`, `encoding`, other pandas options |
| `preview_dataframe()` | Inspect columns + sample | DataFrame | `n`, `random_state` |
| `rename_columns()` | Rename columns | DataFrame, mapping | – |
| `split_column()` | Split into rows or columns | `column`, `delimiter`, `into` | `output_column`, `output_prefix`, `row_id_column`, `drop_empty` |
| `load_location_resources()` | Load countries/GeoNames | – | paths, `locales`, aliases, progress |
| `norm_country()` | Normalize one country | value, `resources` | – |
| `norm_city()` | Normalize one city | value, `resources` | – |
| `process_locations()` | Extract locations and create `affiliation_clean` | DataFrame, `column`, `resources` | output columns, diagnostics, address removal |
| `RORAffiliationProcessor(...)` | Configure ROR workflow | effectively `api_input_col`, `clean_col` | original ROR, country fields, cache, retry/rate settings |
| `generate_exact_candidates()` | Inspect fallback candidates without API | text | `limit` |
| `processor.process()` | Run ROR requests | DataFrame | `statuses`, `max_rows`, `force`, `bypass_cache`, checkpoint, progress |
| `save_csv()` | Save CSV | DataFrame, filename | `overwrite`, CSV options |

The internal helper functions in `locations.py` and `ror_api.py` are not needed for the normal notebook workflow. For everyday use, the public functions documented here should be preferred.

# Recommended workflow

For a new dataset, I would normally proceed as follows:

1. Copy the CSV file into `data/`.
2. Adjust the configuration cell.
3. Load the file and inspect 5–10 rows.
4. Check the affiliation split.
5. Check location cleaning.
6. Optionally use `max_rows=10` for a small API test first.
7. Run the complete API processing.
8. Inspect `status` / `status_detail`.
9. Manually review `review` cases and relevant `unmatched` cases.
10. Save the result.

Do not add a new pipeline rule for every single edge case. For heterogeneous affiliation data, a simple and stable logic with a small number of manual review cases is usually more useful than an increasingly complex rule system.

# Short version for everyday use

Once the tutorial is familiar, the normal workflow essentially reduces to these steps:

In [ ]:
# 1. Load
df = load_csv(PROJECT_ROOT / "data" / DATA_FILENAME)

# 2. Adapt column name
if SOURCE_AFFILIATION_COLUMN != STANDARD_AFFILIATION_COLUMN:
    df = rename_columns(
        df,
        {SOURCE_AFFILIATION_COLUMN: STANDARD_AFFILIATION_COLUMN},
    )

# 3. Split multiple affiliations into separate rows
df_parts = split_column(
    df,
    column=STANDARD_AFFILIATION_COLUMN,
    delimiter=AFFILIATION_DELIMITER,
    into="rows",
    output_column="affiliation_part_raw",
)

# 4. Locations
resources = load_location_resources()

df_clean = process_locations(
    df_parts,
    column="affiliation_part_raw",
    resources=resources,
    include_diagnostics=True,
)

# 5. ROR
processor = RORAffiliationProcessor(
    api_input_col="affiliation_part_raw",
    clean_col="affiliation_clean",
    original_ror_col=(
        ORIGINAL_ROR_COLUMN
        if ORIGINAL_ROR_COLUMN in df_clean.columns
        else None
    ),
    country_code_col="country_iso2",
    country_name_col="country_norm",
    min_request_interval=0.25,
)

df_ror = processor.process(
    df_clean,
    checkpoint_file=CHECKPOINT_FILE,
    show_progress=True,
)

# 6. Save
save_csv(
    df_ror,
    OUTPUT_FILENAME,
    overwrite=True,
)